In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/data-analysis-competition-2025/sample_submission.csv
/kaggle/input/data-analysis-competition-2025/Test/deepseek.csv
/kaggle/input/data-analysis-competition-2025/Test/perplexity.csv
/kaggle/input/data-analysis-competition-2025/Test/claude.csv
/kaggle/input/data-analysis-competition-2025/Test/grok.csv
/kaggle/input/data-analysis-competition-2025/Test/gemini.csv
/kaggle/input/data-analysis-competition-2025/Test/gpt.csv
/kaggle/input/data-analysis-competition-2025/Train/deepseek.csv
/kaggle/input/data-analysis-competition-2025/Train/perplexity.csv
/kaggle/input/data-analysis-competition-2025/Train/claude.csv
/kaggle/input/data-analysis-competition-2025/Train/grok.csv
/kaggle/input/data-analysis-competition-2025/Train/gemini.csv
/kaggle/input/data-analysis-competition-2025/Train/gpt.csv


In [11]:
# === Langkah 0: Instalasi Pustaka ===
# Pastikan pustaka ini sudah terinstal dengan versi terbaru
# !pip install --upgrade transformers datasets accelerate torch scikit-learn emoji

# === Langkah 1: Impor & Persiapan Data Lengkap ===
import pandas as pd
import torch
import glob
import os
import re
import emoji
import numpy as np
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
from sklearn.utils.class_weight import compute_class_weight
from torch.nn import CrossEntropyLoss
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

# Nonaktifkan logging ke Weights & Biases (opsional)
os.environ["WANDB_DISABLED"] = "True"

print("Memulai pemuatan dan pembersihan data...")

# 1A: Muat dan gabungkan semua file CSV dari direktori
path = '/kaggle/input/data-analysis-competition-2025/Train'  # <--- GANTI DENGAN PATH FOLDER ANDA
all_files = glob.glob(os.path.join(path, "*.csv"))
li = [pd.read_csv(f, index_col=None, header=0).assign(source=os.path.basename(f).split('.')[0]) for f in all_files]
frame = pd.concat(li, axis=0, ignore_index=True)
print(f"Total {len(frame)} baris data berhasil dimuat.")

# 1B: Lakukan pembersihan data awal
frame.dropna(subset=['Comment'], inplace=True)
frame['AppVersion'] = frame['AppVersion'].fillna('unknown')
frame.reset_index(drop=True, inplace=True)
print("Pembersihan nilai kosong selesai.")

# 1C: Definisikan dan terapkan fungsi pra-pemrosesan teks
def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    text = emoji.demojize(text, delimiters=(" :", ": "))
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s_:]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

frame['clean_comment'] = frame['Comment'].apply(preprocess_text)
print("Pra-pemrosesan teks (pembuatan 'clean_comment') selesai.")


# === Langkah 2: Mempersiapkan Dataset & Menghitung Bobot Kelas ===

# Pisahkan data menjadi set latih dan uji
train_df, test_df = train_test_split(
    frame,
    test_size=0.2,
    random_state=42,
    stratify=frame['Sentiment']
)

print("\nMenghitung bobot kelas untuk mengatasi data tidak seimbang...")
train_labels = train_df['Sentiment'].values
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)
print(f"Bobot kelas yang dihitung: {class_weights}")
if torch.cuda.is_available():
    class_weights_tensor = class_weights_tensor.to('cuda')

# Ubah pandas DataFrame menjadi Hugging Face Dataset
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

print(f"\nData latih: {len(train_dataset)} baris")
print(f"Data uji: {len(test_dataset)} baris")

# Muat tokenizer dan lakukan tokenisasi
model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(
        examples["clean_comment"], 
        padding="max_length", 
        truncation=True, 
        max_length=128
    )

print("\nMelakukan tokenisasi pada dataset...")
tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_test_dataset = test_dataset.map(tokenize_function, batched=True)

# Ganti nama kolom target menjadi 'labels'
tokenized_train_dataset = tokenized_train_dataset.rename_column("Sentiment", "labels")
tokenized_test_dataset = tokenized_test_dataset.rename_column("Sentiment", "labels")
print("Tokenisasi selesai.")


# === Langkah 3: Membuat Custom Trainer (VERSI PERBAIKAN) ===

class CustomTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get('logits')
        loss_fct = CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# === Langkah 4: Konfigurasi & Proses Fine-Tuning ===
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

training_args = TrainingArguments(
    output_dir="./results_weighted",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    f1 = f1_score(labels, predictions, average="macro")
    accuracy = accuracy_score(labels, predictions)
    return {"f1": f1, "accuracy": accuracy}

trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_test_dataset,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer,
    class_weights=class_weights_tensor
)

print("\nMemulai proses fine-tuning dengan weighted loss...")
trainer.train()
print("Fine-tuning selesai.")


# === Langkah 5: Evaluasi Akhir ===
print("\nMengevaluasi model terbaik pada data uji...")
eval_results = trainer.evaluate()

print("\n--- HASIL EVALUASI AKHIR (DENGAN WEIGHTED LOSS) ---")
print(f"  - Macro F1-Score: {eval_results['eval_f1']:.4f}")
print(f"  - Accuracy: {eval_results['eval_accuracy']:.4f}")
print("-----------------------------------------------------")

Memulai pemuatan dan pembersihan data...
Total 130885 baris data berhasil dimuat.
Pembersihan nilai kosong selesai.
Pra-pemrosesan teks (pembuatan 'clean_comment') selesai.

Menghitung bobot kelas untuk mengatasi data tidak seimbang...
Bobot kelas yang dihitung: [2.49010654 7.8061582  0.40480802]

Data latih: 104704 baris
Data uji: 26176 baris

Melakukan tokenisasi pada dataset...


Map:   0%|          | 0/104704 [00:00<?, ? examples/s]

Map:   0%|          | 0/26176 [00:00<?, ? examples/s]

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Tokenisasi selesai.


/tmp/ipykernel_36/3783728819.py:113: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `CustomTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)



Memulai proses fine-tuning dengan weighted loss...


Epoch,Training Loss,Validation Loss,F1,Accuracy
1,0.772200,0.734754,0.655021,0.895706
2,0.805000,0.808948,0.650600,0.899985


Fine-tuning selesai.

Mengevaluasi model terbaik pada data uji...



--- HASIL EVALUASI AKHIR (DENGAN WEIGHTED LOSS) ---
  - Macro F1-Score: 0.6550
  - Accuracy: 0.8957
-----------------------------------------------------


In [12]:
# ==============================================================================
# LANJUTAN SKRIP... (Setelah trainer.evaluate() selesai)
# ==============================================================================
import numpy as np

# === Langkah 5: Memuat dan Mempersiapkan Data Uji Kompetisi ===

print("\nMemuat dan mempersiapkan data uji untuk submisi...")

# Ganti 'path/to/your/test_folder' dengan lokasi folder 'test' Anda
path_test = '/kaggle/input/data-analysis-competition-2025/Test' 

all_test_files = glob.glob(os.path.join(path_test, "*.csv"))

li_test = []
for f in all_test_files:
    df_temp = pd.read_csv(f, index_col=None, header=0)
    source_name = os.path.basename(f).split('.')[0]
    df_temp['source'] = source_name
    li_test.append(df_temp)

test_competition_df = pd.concat(li_test, axis=0, ignore_index=True)
print(f"Total {len(test_competition_df)} baris data uji berhasil dimuat.")

# Membuat CommentId final yang akurat
test_competition_df['FinalCommentId'] = test_competition_df['source'] + '_' + test_competition_df['CommentId'].astype(str)

# Menerapkan pra-pemrosesan teks yang sama persis
test_competition_df['clean_comment'] = test_competition_df['Comment'].apply(preprocess_text)
print("Pra-pemrosesan data uji selesai.")


# === Langkah 6: Tokenisasi Data Uji untuk Model Transformer ===

# Ubah DataFrame pandas menjadi Dataset Hugging Face
submission_dataset = Dataset.from_pandas(test_competition_df)

# Lakukan tokenisasi menggunakan fungsi yang sama
print("\nMelakukan tokenisasi pada data uji...")
tokenized_submission_dataset = submission_dataset.map(tokenize_function, batched=True)
print("Tokenisasi selesai.")


# === Langkah 7: Membuat Prediksi dengan Trainer ===

print("\nMembuat prediksi menggunakan model Transformer...")
# Gunakan trainer.predict() untuk mendapatkan output mentah (logits)
raw_predictions = trainer.predict(tokenized_submission_dataset)

# Ambil kelas dengan nilai tertinggi dari logits
# raw_predictions.predictions adalah array numpy dari logits
final_predictions = np.argmax(raw_predictions.predictions, axis=-1)


# === Langkah 8: Membuat dan Menyimpan File Submisi Final ===

# Buat DataFrame submisi dengan format yang benar
submission_df = pd.DataFrame({
    'CommentId': test_competition_df['FinalCommentId'],
    'Sentiment': final_predictions
})

# Simpan ke file CSV
submission_df.to_csv('/kaggle/working/results/submission_transformer.csv', index=False)

print("\nFile 'submission_transformer.csv' berhasil dibuat!")
print("Ini adalah file submisi utama kita. Silakan unggah ke Kaggle.")
print("Contoh isi file submisi:")
print(submission_df.head())


Memuat dan mempersiapkan data uji untuk submisi...
Total 45971 baris data uji berhasil dimuat.
Pra-pemrosesan data uji selesai.

Melakukan tokenisasi pada data uji...


Map:   0%|          | 0/45971 [00:00<?, ? examples/s]

Tokenisasi selesai.

Membuat prediksi menggunakan model Transformer...

File 'submission_transformer.csv' berhasil dibuat!
Ini adalah file submisi utama kita. Silakan unggah ke Kaggle.
Contoh isi file submisi:
    CommentId  Sentiment
0  deepseek_1          2
1  deepseek_2          2
2  deepseek_3          1
3  deepseek_4          2
4  deepseek_5          1
